In [1]:
from pathlib import Path
import getpass
import os
import shutil

import numpy as np
import pandas as pd

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

C:\Users\ASUS\AppData\Local\Temp\ipykernel_27620\2860160806.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\ASUS\Desktop\Gen AI Full stak\env\Lib\site-packages\cupy\_environment.py:284: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(
c:\Users\ASUS\Desktop\Gen AI Full stak\env\Lib\site-packages\cupy\_environment.py:284: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


In [2]:
import os
import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass(
        "Enter your Groq API key: "
    )

print("Groq API key configured successfully.")

Groq API key configured successfully.


In [3]:
from pathlib import Path

PDF_PATH = Path(
    r"C:\Users\ASUS\Desktop\Gen AI Full stak\class_23_chuncking\data\llama2-research-paper.pdf"
)

if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"PDF file was not found:\n{PDF_PATH}"
    )

print("PDF found:", PDF_PATH)

PDF found: C:\Users\ASUS\Desktop\Gen AI Full stak\class_23_chuncking\data\llama2-research-paper.pdf


In [4]:
loader = PyPDFLoader(str(PDF_PATH))

pages = loader.load()

print(f"Total PDF pages loaded: {len(pages)}")

Total PDF pages loaded: 77


In [5]:
print("First-page metadata:")
print(pages[0].metadata)

print("\nFirst 1,000 characters:")
print(pages[0].page_content[:1000])

First-page metadata:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'C:\\Users\\ASUS\\Desktop\\Gen AI Full stak\\class_23_chuncking\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}

First 1,000 characters:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez M

In [7]:
def identify_section(paper_page: int) -> str:
    """
    Identify the major section of the Llama 2 paper
    using its printed PDF page number.
    """

    if 1 <= paper_page <= 2:
        return "front_matter"

    if 3 <= paper_page <= 4:
        return "introduction"

    if 5 <= paper_page <= 7:
        return "pretraining"

    if 8 <= paper_page <= 19:
        return "fine_tuning"

    if 20 <= paper_page <= 31:
        return "safety"

    if 32 <= paper_page <= 35:
        return "discussion"

    if paper_page == 36:
        return "conclusion"

    if 37 <= paper_page <= 45:
        return "references"

    if 46 <= paper_page <= 77:
        return "appendix"

    return "unknown"

In [8]:
for page_document in pages:
    # PyPDFLoader page index is normally zero-based
    page_index = int(page_document.metadata.get("page", 0))
    paper_page = page_index + 1

    page_document.metadata.update(
        {
            "paper": "Llama 2",
            "organization": "Meta",
            "year": 2023,
            "document_type": "research_paper",
            "paper_page": paper_page,
            "section": identify_section(paper_page),
            "access_level": "public",
        }
    )

In [9]:
for page_document in pages[:5]:
    print(page_document.metadata)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'C:\\Users\\ASUS\\Desktop\\Gen AI Full stak\\class_23_chuncking\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'so

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-36-29-July-2026-Retriever\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}


In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = text_splitter.split_documents(pages)

print(f"Total pages: {len(pages)}")
print(f"Total chunks: {len(chunks)}")

Total pages: 77
Total chunks: 343


In [11]:
for chunk_number, chunk in enumerate(chunks):
    paper_page = chunk.metadata.get("paper_page", "unknown")

    chunk.metadata["chunk_id"] = (
        f"llama2-page-{paper_page}-chunk-{chunk_number}"
    )

In [12]:
print("Chunk content:")
print(chunks[0].page_content[:1000])

print("\nChunk metadata:")
print(chunks[0].metadata)

Chunk content:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojni

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

In [14]:
test_vector = embeddings.embed_query(
    "What is Llama 2?"
)

print(f"Embedding dimensions: {len(test_vector)}")
print(f"First 10 values: {test_vector[:10]}")

Embedding dimensions: 768
First 10 values: [-0.031806789338588715, -0.007344976533204317, 0.004927697125822306, -0.0058638653717935085, 0.03312014043331146, -0.03700491786003113, 0.015019138343632221, 0.02953345701098442, -0.007648929487913847, 0.0010430555557832122]


In [18]:
from pathlib import Path
import shutil

RETRIEVAL_DIR = Path(
    r"C:\Users\ASUS\Desktop\Gen AI Full stak\class_24_retrievel"
)

PERSIST_DIRECTORY = RETRIEVAL_DIR / "chroma_llama2_retriever"

REBUILD_INDEX = True

if REBUILD_INDEX and PERSIST_DIRECTORY.exists():
    shutil.rmtree(
        PERSIST_DIRECTORY,
        ignore_errors=True
    )

print(PERSIST_DIRECTORY)

C:\Users\ASUS\Desktop\Gen AI Full stak\class_24_retrievel\chroma_llama2_retriever


In [16]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="llama2_retriever_demo",
    persist_directory=str(PERSIST_DIRECTORY),
    collection_configuration={
        "hnsw": {
            "space": "cosine"
        }
    },
)

print("Vector store created successfully.")
print(f"Stored chunks: {len(chunks)}")
print(f"Persisted at: {PERSIST_DIRECTORY}")

Vector store created successfully.
Stored chunks: 343
Persisted at: C:\Users\ASUS\Desktop\Gen AI Full stak\class_24_retrievel\chroma_llama2_retriever


In [19]:
from pathlib import Path

RETRIEVAL_DIR = Path(
    r"C:\Users\ASUS\Desktop\Gen AI Full stak\class_24_retrievel"
)

PERSIST_DIRECTORY = RETRIEVAL_DIR / "chroma_llama2_retriever"

print(PERSIST_DIRECTORY)

C:\Users\ASUS\Desktop\Gen AI Full stak\class_24_retrievel\chroma_llama2_retriever


In [20]:
PERSIST_DIRECTORY

WindowsPath('C:/Users/ASUS/Desktop/Gen AI Full stak/class_24_retrievel/chroma_llama2_retriever')

In [21]:
# Load the existing Chroma collection
vector_store = Chroma(
    collection_name="llama2_retriever_demo",
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIRECTORY),
)

print("Existing vector store loaded successfully.")
print(f"Persist directory: {PERSIST_DIRECTORY}")

Existing vector store loaded successfully.
Persist directory: C:\Users\ASUS\Desktop\Gen AI Full stak\class_24_retrievel\chroma_llama2_retriever


In [22]:
def display_documents(
    documents,
    max_characters: int = 700
) -> None:
    """
    Display retrieved LangChain Document objects clearly.
    """

    if not documents:
        print("No documents were returned.")
        return

    for rank, document in enumerate(documents, start=1):
        metadata = document.metadata

        print("=" * 90)
        print(f"RANK: {rank}")
        print(f"PAPER PAGE: {metadata.get('paper_page')}")
        print(f"SECTION: {metadata.get('section')}")
        print(f"CHUNK ID: {metadata.get('chunk_id')}")
        print(f"SOURCE: {metadata.get('source')}")
        print("-" * 90)
        print(document.page_content[:max_characters])
        print()

In [ ]:
# vector_store.similarity_search()

In [23]:
similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    }
)

In [24]:
query = "What model sizes of Llama 2 were released?"

similarity_documents = similarity_retriever.invoke(query)

display_documents(similarity_documents)

RANK: 1
PAPER PAGE: 77
SECTION: appendix
CHUNK ID: llama2-page-77-chunk-338
SOURCE: C:\Users\ASUS\Desktop\Gen AI Full stak\class_23_chuncking\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
A.7 Model Card
Table 52 presents a model card (Mitchell et al., 2018; Anil et al., 2023) that summarizes details of the models.
Model Details
Model DevelopersMeta AI
Variations Llama 2comes in a range of parameter sizes—7B, 13B, and 70B—as well as
pretrained and fine-tuned variations.
Input Models input text only.
Output Models generate text only.
Model ArchitectureLlama 2isanauto-regressivelanguagemodelthatusesanoptimizedtransformer
architecture. The tuned versions use supervised fine-tuning (SFT) and reinforce-
ment learning with human feedback (RLHF) to align to human preferences for
helpfulness and safety.
Model Dates Llama 2was trained between January 2023 and July 2023.
Status This is 

RANK: 2
PAPER PAGE: 50
SECTION: ap

In [26]:
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    }
)

In [27]:
query = "How was Llama 2-Chat trained and aligned?"

mmr_documents = mmr_retriever.invoke(query)

display_documents(mmr_documents)

RANK: 1
PAPER PAGE: 8
SECTION: fine_tuning
CHUNK ID: llama2-page-8-chunk-29
SOURCE: C:\Users\ASUS\Desktop\Gen AI Full stak\class_23_chuncking\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
are from OpenAI (2023). Results for the PaLM model are from Chowdhery et al. (2022). Results for the
PaLM-2-L are from Anil et al. (2023).
3 Fine-tuning
Llama 2-Chat is the result of several months of research and iterative applications of alignment techniques,
including both instruction tuning and RLHF, requiring significant computational and annotation resources.
In this section, we report on our experiments and findings using supervised fine-tuning (Section 3.1), as
well as initial and iterative reward modeling (Section 3.2.2) and RLHF (Section 3.2.3). We also share a
new technique, Ghost Attention (GAtt), which we find helps control dialogue flow over multiple turns
(Section 3.3). See Se

RANK: 2
PAPER PAGE: 54
SECTION: ap

In [28]:
query = "How was Llama 2-Chat trained and aligned?"

similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.5,
    }
)

similarity_results = similarity_retriever.invoke(query)
mmr_results = mmr_retriever.invoke(query)

In [29]:
print("Similarity Search results:")
for document in similarity_results:
    print(
        document.metadata.get("paper_page"),
        document.metadata.get("section"),
        document.metadata.get("chunk_id"),
    )

print("\nMMR results:")
for document in mmr_results:
    print(
        document.metadata.get("paper_page"),
        document.metadata.get("section"),
        document.metadata.get("chunk_id"),
    )

Similarity Search results:
8 fine_tuning llama2-page-8-chunk-29
18 fine_tuning llama2-page-18-chunk-77
4 introduction llama2-page-4-chunk-13
4 introduction llama2-page-4-chunk-12

MMR results:
8 fine_tuning llama2-page-8-chunk-29
54 appendix llama2-page-54-chunk-239
34 discussion llama2-page-34-chunk-147
56 appendix llama2-page-56-chunk-247


In [30]:
threshold_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 10,
        "score_threshold": 0.64,
    }
)

In [31]:
query = "What safety techniques were used for Llama 2-Chat?"

threshold_documents = threshold_retriever.invoke(query)

display_documents(threshold_documents)

RANK: 1
PAPER PAGE: 4
SECTION: introduction
CHUNK ID: llama2-page-4-chunk-13
SOURCE: C:\Users\ASUS\Desktop\Gen AI Full stak\class_23_chuncking\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
Solaiman et al., 2023). Testing conducted to date has been in English and has not — and could not — cover
all scenarios. Therefore, before deploying any applications ofLlama 2-Chat, developers should perform
safety testing and tuning tailored to their specific applications of the model. We provide a responsible use
guide¶ and code examples‖ to facilitate the safe deployment ofLlama 2 and Llama 2-Chat. More details of
our responsible release strategy can be found in Section 5.3.
The remainder of this paper describes our pretraining methodology (Section 2), fine-tuning methodology
(Section 3), approach to model safety (Section 4), key observations and insights (Section 5), relevant related
wor

RANK: 2
PAPER PAGE: 4
SECTION: in

In [32]:
# threshold_retriever = vector_store.as_retriever(
#     search_type="similarity_score_threshold",
#     search_kwargs={
#         "k": 10,
#         "score_threshold": 0.35,
#     }


In [33]:
query = "What safety techniques were used for Llama 2-Chat?"

scored_results = vector_store.similarity_search_with_relevance_scores(
    query=query,
    k=5,
)

for rank, (document, relevance_score) in enumerate(
    scored_results,
    start=1
):
    print("=" * 90)
    print(f"Rank: {rank}")
    print(f"Relevance score: {relevance_score:.4f}")
    print(f"Paper page: {document.metadata.get('paper_page')}")
    print(f"Section: {document.metadata.get('section')}")
    print(document.page_content[:500])

Rank: 1
Relevance score: 0.8076
Paper page: 4
Section: introduction
Solaiman et al., 2023). Testing conducted to date has been in English and has not — and could not — cover
all scenarios. Therefore, before deploying any applications ofLlama 2-Chat, developers should perform
safety testing and tuning tailored to their specific applications of the model. We provide a responsible use
guide¶ and code examples‖ to facilitate the safe deployment ofLlama 2 and Llama 2-Chat. More details of
our responsible release strategy can be found in Section 5.3.
The remainder of 
Rank: 2
Relevance score: 0.7312
Paper page: 4
Section: introduction
Figure 3: Safety human evaluation results forLlama 2-Chat compared to other open-source and closed-
source models. Human raters judged model generations for safety violations across ~2,000 adversarial
prompts consisting of both single and multi-turn prompts. More details can be found in Section 4.4. It is
important to caveat these safety results with the inhere

In [34]:
metric_query = "How was reinforcement learning with human feedback used?"

candidate_documents = vector_store.similarity_search(
    metric_query,
    k=6,
)

candidate_texts = [
    document.page_content
    for document in candidate_documents
]

print(f"Candidate chunks selected: {len(candidate_texts)}")

Candidate chunks selected: 6


In [35]:
query_vector = np.asarray(
    embeddings.embed_query(metric_query),
    dtype=np.float64,
)

document_vectors = np.asarray(
    embeddings.embed_documents(candidate_texts),
    dtype=np.float64,
)

print("Query-vector shape:", query_vector.shape)
print("Document-vectors shape:", document_vectors.shape)

Query-vector shape: (768,)
Document-vectors shape: (6, 768)


In [36]:
def cosine_similarity(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    denominator = (
        np.linalg.norm(vector_a)
        * np.linalg.norm(vector_b)
    )

    if denominator == 0:
        return 0.0

    return float(
        np.dot(vector_a, vector_b) / denominator
    )


def euclidean_distance(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    return float(
        np.linalg.norm(vector_a - vector_b)
    )


def dot_product(
    vector_a: np.ndarray,
    vector_b: np.ndarray
) -> float:
    return float(
        np.dot(vector_a, vector_b)
    )

In [37]:
metric_rows = []

for document, document_vector in zip(
    candidate_documents,
    document_vectors
):
    metric_rows.append(
        {
            "paper_page": document.metadata.get("paper_page"),
            "section": document.metadata.get("section"),
            "chunk_id": document.metadata.get("chunk_id"),
            "cosine_similarity": cosine_similarity(
                query_vector,
                document_vector
            ),
            "euclidean_distance": euclidean_distance(
                query_vector,
                document_vector
            ),
            "dot_product": dot_product(
                query_vector,
                document_vector
            ),
            "preview": document.page_content[:100].replace(
                "\n",
                " "
            ),
        }
    )

metric_table = pd.DataFrame(metric_rows)

metric_table

,paper_page,section,chunk_id,cosine_similarity,euclidean_distance,dot_product,preview
0,5,pretraining,llama2-page-5-chunk-14,0.759080,0.694147,0.759080,Figure 4: Training ofLlama 2-Chat: This proces...
1,2,front_matter,llama2-page-2-chunk-3,0.748611,0.709068,0.748611,Contents 1 Introduction 3 2 Pretraining 5 2.1 ...
2,10,fine_tuning,llama2-page-10-chunk-35,0.741688,0.718766,0.741688,"sampled human preferences, whereby human annot..."
3,9,fine_tuning,llama2-page-9-chunk-34,0.732295,0.731717,0.732295,"learning rate of2 × 10−5, a weight decay of 0...."
4,36,conclusion,llama2-page-36-chunk-157,0.714277,0.755940,0.714277,"In this paradigm, models are fine-tuned based ..."
5,32,discussion,llama2-page-32-chunk-138,0.711838,0.759161,0.711838,"bility, seemed a somewhat shadowy field for th..."


In [38]:
metric_query

'How was reinforcement learning with human feedback used?'

In [39]:
metric_table.sort_values(
    by="cosine_similarity",
    ascending=False
)[
    [
        "paper_page",
        "section",
        "cosine_similarity",
        "preview",
    ]
]

,paper_page,section,cosine_similarity,preview
0,5,pretraining,0.759080,Figure 4: Training ofLlama 2-Chat: This proces...
1,2,front_matter,0.748611,Contents 1 Introduction 3 2 Pretraining 5 2.1 ...
2,10,fine_tuning,0.741688,"sampled human preferences, whereby human annot..."
3,9,fine_tuning,0.732295,"learning rate of2 × 10−5, a weight decay of 0...."
4,36,conclusion,0.714277,"In this paradigm, models are fine-tuned based ..."
5,32,discussion,0.711838,"bility, seemed a somewhat shadowy field for th..."


In [40]:
metric_table.sort_values(
    by="euclidean_distance",
    ascending=True
)[
    [
        "paper_page",
        "section",
        "euclidean_distance",
        "preview",
    ]
]

,paper_page,section,euclidean_distance,preview
0,5,pretraining,0.694147,Figure 4: Training ofLlama 2-Chat: This proces...
1,2,front_matter,0.709068,Contents 1 Introduction 3 2 Pretraining 5 2.1 ...
2,10,fine_tuning,0.718766,"sampled human preferences, whereby human annot..."
3,9,fine_tuning,0.731717,"learning rate of2 × 10−5, a weight decay of 0...."
4,36,conclusion,0.755940,"In this paradigm, models are fine-tuned based ..."
5,32,discussion,0.759161,"bility, seemed a somewhat shadowy field for th..."


In [41]:
metric_table.sort_values(
    by="dot_product",
    ascending=False
)[
    [
        "paper_page",
        "section",
        "dot_product",
        "preview",
    ]
]

,paper_page,section,dot_product,preview
0,5,pretraining,0.759080,Figure 4: Training ofLlama 2-Chat: This proces...
1,2,front_matter,0.748611,Contents 1 Introduction 3 2 Pretraining 5 2.1 ...
2,10,fine_tuning,0.741688,"sampled human preferences, whereby human annot..."
3,9,fine_tuning,0.732295,"learning rate of2 × 10−5, a weight decay of 0...."
4,36,conclusion,0.714277,"In this paradigm, models are fine-tuned based ..."
5,32,discussion,0.711838,"bility, seemed a somewhat shadowy field for th..."


In [42]:
normalized_query_vector = (
    query_vector / np.linalg.norm(query_vector)
)

normalized_document_vectors = (
    document_vectors
    / np.linalg.norm(
        document_vectors,
        axis=1,
        keepdims=True
    )
)

normalized_dot_scores = (
    normalized_document_vectors
    @ normalized_query_vector
)

cosine_scores = np.asarray(
    [
        cosine_similarity(
            query_vector,
            document_vector
        )
        for document_vector in document_vectors
    ]
)

print("Cosine scores:")
print(cosine_scores)

print("\nDot product after normalization:")
print(normalized_dot_scores)

print(
    "\nAre they approximately equal?",
    np.allclose(
        cosine_scores,
        normalized_dot_scores,
        atol=1e-8,
    ),
)

Cosine scores:
[0.75907992 0.74861139 0.74168801 0.73229509 0.71427736 0.71183763]

Dot product after normalization:
[0.75907992 0.74861139 0.74168801 0.73229509 0.71427736 0.71183763]

Are they approximately equal? True


In [43]:
fine_tuning_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {
            "section": "fine_tuning"
        },
    }
)

In [44]:
query = "How was Llama 2-Chat aligned with human preferences?"

fine_tuning_documents = fine_tuning_retriever.invoke(query)

display_documents(fine_tuning_documents)

RANK: 1
PAPER PAGE: 18
SECTION: fine_tuning
CHUNK ID: llama2-page-18-chunk-77
SOURCE: C:\Users\ASUS\Desktop\Gen AI Full stak\class_23_chuncking\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
biased in favor ofLlama 2-Chat. Therefore, for a fair comparison, we additionally compute the final results
usingGPT-4toassesswhichgenerationispreferred. TheorderinwhichChatGPTand Llama 2-Chatoutputs
appeared in GPT-4 prompt are randomly swapped to avoid any bias. As expected, the win-rate in favor of
Llama 2-Chat is less pronounced, although obtaining more than a 60% win-rate for our latestLlama 2-Chat.
The prompts correspond to a validation set of1, 586and 584prompts for safety and helpfulness, respectively.
3.4.2 Human Evaluation
Human evaluation is often considered the gold standard for judging models for natural language generation,
including dialogue models. To evaluate the quality of 

RANK: 2
PAPER PAGE: 10
SECTION: 

In [45]:
for document in fine_tuning_documents:
    assert document.metadata["section"] == "fine_tuning"

print("All returned documents are from the fine_tuning section.")

All returned documents are from the fine_tuning section.


In [46]:
filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": {
            "$and": [
                {
                    "section": {
                        "$eq": "fine_tuning"
                    }
                },
                {
                    "year": {
                        "$eq": 2023
                    }
                },
                {
                    "organization": {
                        "$eq": "Meta"
                    }
                },
            ]
        },
    }
)

In [47]:
query = "How was human preference data collected?"

filtered_documents = filtered_retriever.invoke(query)

display_documents(filtered_documents)

RANK: 1
PAPER PAGE: 10
SECTION: fine_tuning
CHUNK ID: llama2-page-10-chunk-35
SOURCE: C:\Users\ASUS\Desktop\Gen AI Full stak\class_23_chuncking\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
sampled human preferences, whereby human annotators select which of two model outputs they prefer.
This human feedback is subsequently used to train a reward model, which learns patterns in the preferences
of the human annotators and can then automate preference decisions.
3.2.1 Human Preference Data Collection
Next, we collect human preference data for reward modeling. We chose a binary comparison protocol over
other schemes, mainly because it enables us to maximize the diversity of collected prompts. Still, other
strategies are worth considering, which we leave for future work.
Our annotation procedure proceeds as follows. We ask annotators to first write a prompt, then choose
between two 

RANK: 2
PAPER PAGE: 11
SECTION: 

Prefilter

In [48]:
pre_filter = {
    "$and": [
        {
            "section": {
                "$eq": "fine_tuning"
            }
        },
        {
            "year": {
                "$eq": 2023
            }
        },
    ]
}

pre_filtered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4,
        "filter": pre_filter,
    }
)

pre_filtered_documents = pre_filtered_retriever.invoke(
    "How was the reward model trained?"
)

display_documents(pre_filtered_documents)

RANK: 1
PAPER PAGE: 10
SECTION: fine_tuning
CHUNK ID: llama2-page-10-chunk-40
SOURCE: C:\Users\ASUS\Desktop\Gen AI Full stak\class_23_chuncking\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
3.2.2 Reward Modeling
The reward model takes a model response and its corresponding prompt (including contexts from previous
turns) as inputs and outputs a scalar score to indicate the quality (e.g., helpfulness and safety) of the model
generation. Leveraging such response scores as rewards, we can optimizeLlama 2-Chat during RLHF for
better human preference alignment and improved helpfulness and safety.
Others have found that helpfulness and safety sometimes trade off (Bai et al., 2022a), which can make it
challenging for a single reward model to perform well on both. To address this, we train two separate reward
models, one optimized for helpfulness (referred to asHelpfulness RM) and anoth

RANK: 2
PAPER PAGE: 13
SECTION: 

Post-Filtering

In [49]:
unfiltered_candidates = vector_store.similarity_search(
    query="How was the reward model trained?",
    k=15,
)

In [50]:
post_filtered_documents = [
    document
    for document in unfiltered_candidates
    if document.metadata.get("section") == "fine_tuning"
    and document.metadata.get("year") == 2023
]

display_documents(post_filtered_documents)

RANK: 1
PAPER PAGE: 10
SECTION: fine_tuning
CHUNK ID: llama2-page-10-chunk-40
SOURCE: C:\Users\ASUS\Desktop\Gen AI Full stak\class_23_chuncking\data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
3.2.2 Reward Modeling
The reward model takes a model response and its corresponding prompt (including contexts from previous
turns) as inputs and outputs a scalar score to indicate the quality (e.g., helpfulness and safety) of the model
generation. Leveraging such response scores as rewards, we can optimizeLlama 2-Chat during RLHF for
better human preference alignment and improved helpfulness and safety.
Others have found that helpfulness and safety sometimes trade off (Bai et al., 2022a), which can make it
challenging for a single reward model to perform well on both. To address this, we train two separate reward
models, one optimized for helpfulness (referred to asHelpfulness RM) and anoth

RANK: 2
PAPER PAGE: 13
SECTION: 

In [51]:
print(
    "Documents retrieved before post-filtering:",
    len(unfiltered_candidates),
)

print(
    "Documents remaining after post-filtering:",
    len(post_filtered_documents),
)

Documents retrieved before post-filtering: 15
Documents remaining after post-filtering: 11


In [56]:


from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

question = "What are the main components of Llama 2?"

docs = vector_store.similarity_search(
    question,
    k=3
)

context = "\n\n".join(
    doc.page_content for doc in docs
)

prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{question}
"""

response = llm.invoke(prompt)

print(response.content)

The model card for Llama 2 lists the following key components:

| Component | Details (as given in the card) |
|-----------|--------------------------------|
| **Model Developers** | Meta AI |
| **Variations** | Llama 2 is released in several parameter sizes – 7 B, 13 B, and 70 B – and includes both pretrained and fine‑tuned versions. |
| **Input** | The model accepts **text only**. |
| **Output** | The model generates **text only**. |
| **Model Architecture** | An **auto‑regressive language model** that uses an optimized Transformer architecture. The fine‑tuned versions are trained with supervised fine‑tuning (SFT) and reinforcement learning with human feedback (RLHF) to improve helpfulness and safety. |
| **Model Dates** | Trained between **January 2023 and July 2023**. |
| **Status** | A **static model** trained on an offline dataset; future tuned versions will be released as safety improvements are made. |
| **License** | Available under a **custom commercial license** (see ai.meta

05-Aug-2026

In [57]:
import os
from typing import List

from pydantic import BaseModel, Field

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_classic.retrievers.contextual_compression import (
    ContextualCompressionRetriever,
)
from langchain_classic.retrievers.document_compressors import (
    CrossEncoderReranker,
)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

In [58]:
def display_documents(
    documents,
    title: str = "Retrieved Documents",
    max_documents: int = 10,
    max_characters: int = 600,
) -> None:
    """Display retrieved LangChain Document objects."""

    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    if not documents:
        print("No documents were returned.")
        return

    for rank, document in enumerate(
        documents[:max_documents],
        start=1,
    ):
        metadata = document.metadata

        print(f"\nRANK: {rank}")
        print(f"Paper page: {metadata.get('paper_page')}")
        print(f"Section: {metadata.get('section')}")
        print(f"Chunk ID: {metadata.get('chunk_id')}")
        print("-" * 100)
        print(document.page_content[:max_characters])

In [59]:
def deduplicate_documents(documents):
    """Remove duplicate retrieved chunks while preserving their order."""

    unique_documents = []
    seen_keys = set()

    for document in documents:
        key = (
            document.metadata.get("chunk_id")
            or (
                document.metadata.get("source"),
                document.metadata.get("page"),
                document.page_content,
            )
        )

        if key not in seen_keys:
            seen_keys.add(key)
            unique_documents.append(document)

    return unique_documents

In [60]:
from langchain_community.retrievers import BM25Retriever

In [61]:
bm25_retriever = BM25Retriever.from_documents(chunks)

In [62]:
bm25_retriever

BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001AF30EB1410>)

In [63]:
# Final number of results
bm25_retriever.k = 4

In [64]:
sparse_query = "Grouped-Query Attention GQA 70B"

In [65]:
sparse_documents = bm25_retriever.invoke(sparse_query)

In [66]:
display_documents(
    sparse_documents,
    title="Sparse Retrieval: BM25 Results",
)


Sparse Retrieval: BM25 Results

RANK: 1
Paper page: 6
Section: pretraining
Chunk ID: llama2-page-6-chunk-18
----------------------------------------------------------------------------------------------------
Training Data Params Context
Length
GQA Tokens LR
Llama 1 See Touvron et al.
(2023)
7B 2k ✗ 1.0T 3.0 × 10−4
13B 2k ✗ 1.0T 3.0 × 10−4
33B 2k ✗ 1.4T 1.5 × 10−4
65B 2k ✗ 1.4T 1.5 × 10−4
Llama 2 A new mix of publicly
available online data
7B 4k ✗ 2.0T 3.0 × 10−4
13B 4k ✗ 2.0T 3.0 × 10−4
34B 4k ✓ 2.0T 1.5 × 10−4
70B 4k ✓ 2.0T 1.5 × 10−4
Table 1:Llama 2 family of models.Token counts refer to pretraining data only. All models are trained with
a global batch-size of 4M tokens. Bigger models — 34B and 70B — use Grouped-Query Attention (GQA) for
improved inference scalability.
0 250 500 750 1000 1250 15

RANK: 2
Paper page: 48
Section: appendix
Chunk ID: llama2-page-48-chunk-220
----------------------------------------------------------------------------------------------------
BoolQ PIQA 

In [67]:
dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    },
)

In [68]:
dense_query = (
    "How did Meta improve inference scalability "
    "for the largest Llama 2 models?"
)

dense_documents = dense_retriever.invoke(dense_query)

display_documents(
    dense_documents,
    title="Dense Retrieval: Vector Search Results",
)


Dense Retrieval: Vector Search Results

RANK: 1
Paper page: 77
Section: appendix
Chunk ID: llama2-page-77-chunk-338
----------------------------------------------------------------------------------------------------
A.7 Model Card
Table 52 presents a model card (Mitchell et al., 2018; Anil et al., 2023) that summarizes details of the models.
Model Details
Model DevelopersMeta AI
Variations Llama 2comes in a range of parameter sizes—7B, 13B, and 70B—as well as
pretrained and fine-tuned variations.
Input Models input text only.
Output Models generate text only.
Model ArchitectureLlama 2isanauto-regressivelanguagemodelthatusesanoptimizedtransformer
architecture. The tuned versions use supervised fine-tuning (SFT) and reinforce-
ment learning with human feedback (RLHF) to align to human preferences for
helpfu

RANK: 2
Paper page: 6
Section: pretraining
Chunk ID: llama2-page-6-chunk-18
----------------------------------------------------------------------------------------------------
Tra

In [69]:
comparison_query = (
    "How did grouped-query attention improve "
    "Llama 2 inference scalability?"
)

sparse_results = bm25_retriever.invoke(comparison_query)
dense_results = dense_retriever.invoke(comparison_query)

display_documents(
    sparse_results,
    title="BM25 Results",
    max_documents=4,
)

display_documents(
    dense_results,
    title="Dense Vector Results",
    max_documents=4,
)


BM25 Results

RANK: 1
Paper page: 4
Section: introduction
Chunk ID: llama2-page-4-chunk-12
----------------------------------------------------------------------------------------------------
1. Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data. We also
increased the size of the pretraining corpus by 40%, doubled the context length of the model, and
adopted grouped-query attention (Ainslie et al., 2023). We are releasing variants ofLlama 2 with
7B, 13B, and 70B parameters. We have also trained 34B variants, which we report on in this paper
but are not releasing.§
2. Llama 2-Chat, a fine-tuned version ofLlama 2 that is optimized for dialogue use cases. We release
variants of this model with 7B, 13B, and 70B parameters as well.
We believe that the

RANK: 2
Paper page: 47
Section: appendix
Chunk ID: llama2-page-47-chunk-217
----------------------------------------------------------------------------------------------------
attention (MHA) models grow 

In [71]:
print("SPARSE RESULTS")
for rank, document in enumerate(sparse_results, start=1):
    print(
        rank,
        document.metadata.get("paper_page"),
        document.metadata.get("chunk_id"),
    )

print("\nDENSE RESULTS")
for rank, document in enumerate(dense_results, start=1):
    print(
        rank,
        document.metadata.get("paper_page"),
        document.metadata.get("chunk_id"),
    )

SPARSE RESULTS
1 4 llama2-page-4-chunk-12
2 47 llama2-page-47-chunk-217
3 54 llama2-page-54-chunk-240
4 6 llama2-page-6-chunk-18

DENSE RESULTS
1 6 llama2-page-6-chunk-18
2 47 llama2-page-47-chunk-217
3 4 llama2-page-4-chunk-12
4 13 llama2-page-13-chunk-53


In [72]:
bm25_retriever.k = 8

dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 8
    },
)

In [73]:
from langchain_classic.retrievers.ensemble import EnsembleRetriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        dense_retriever,
    ],
    weights=[
        0.5,  # BM25 weight
        0.5,  # Dense-retrieval weight
    ],
)

In [74]:
hybrid_query = (
    "Llama 2 70B grouped-query attention and inference scalability"
)

In [75]:
hybrid_documents = hybrid_retriever.invoke(hybrid_query)

In [76]:
display_documents(
    hybrid_documents,
    title="Hybrid Retrieval: BM25 + Dense + RRF",
    max_documents=6,
)


Hybrid Retrieval: BM25 + Dense + RRF

RANK: 1
Paper page: 6
Section: pretraining
Chunk ID: llama2-page-6-chunk-18
----------------------------------------------------------------------------------------------------
Training Data Params Context
Length
GQA Tokens LR
Llama 1 See Touvron et al.
(2023)
7B 2k ✗ 1.0T 3.0 × 10−4
13B 2k ✗ 1.0T 3.0 × 10−4
33B 2k ✗ 1.4T 1.5 × 10−4
65B 2k ✗ 1.4T 1.5 × 10−4
Llama 2 A new mix of publicly
available online data
7B 4k ✗ 2.0T 3.0 × 10−4
13B 4k ✗ 2.0T 3.0 × 10−4
34B 4k ✓ 2.0T 1.5 × 10−4
70B 4k ✓ 2.0T 1.5 × 10−4
Table 1:Llama 2 family of models.Token counts refer to pretraining data only. All models are trained with
a global batch-size of 4M tokens. Bigger models — 34B and 70B — use Grouped-Query Attention (GQA) for
improved inference scalability.
0 250 500 750 1000 1250 15

RANK: 2
Paper page: 47
Section: appendix
Chunk ID: llama2-page-47-chunk-217
----------------------------------------------------------------------------------------------------
atten

                         ┌── BM25 Retriever ─────┐
User query ──────────────┤                       ├── Weighted RRF
                         └── Dense Retriever ────┘
                                                    ↓
                                            Combined ranking

In [77]:
test_query = (
    "How did Meta make Llama 2 70B efficient for large-scale inference?"
)

In [78]:
sparse_results = bm25_retriever.invoke(test_query)
dense_results = dense_retriever.invoke(test_query)
hybrid_results = hybrid_retriever.invoke(test_query)

In [79]:
display_documents(
    sparse_results,
    title="1. Sparse Retrieval",
    max_documents=4,
)

display_documents(
    dense_results,
    title="2. Dense Retrieval",
    max_documents=4,
)

display_documents(
    hybrid_results,
    title="3. Hybrid Retrieval",
    max_documents=4,
)


1. Sparse Retrieval

RANK: 1
Paper page: 54
Section: appendix
Chunk ID: llama2-page-54-chunk-240
----------------------------------------------------------------------------------------------------
attribute, and so, up to 20 turns (we did not extend the human evaluation more, and all the examples had
less than 4048 tokens in total over the turns). As a comparison,Llama 2-Chat without GAtt can not anymore
refer to the attributes after only few turns: from 100% at turn t+1, to 10% at turn t+3 and then 0%.
GAtt Zero-shot Generalisation. We tried at inference time to set constrain not present in the training of
GAtt. For instance, “answer in one sentence only”, for which the model remained consistent, as illustrated in
Figure 28.
We applied first GAtt toLlama 1, which was pretrained with a 

RANK: 2
Paper page: 6
Section: pretraining
Chunk ID: llama2-page-6-chunk-18
----------------------------------------------------------------------------------------------------
Training Data Params C

# Query Decomposition

In [83]:
import os
from langchain_groq import ChatGroq

CHAT_MODEL = os.environ.get(
    "GROQ_CHAT_MODEL",
    "openai/gpt-oss-120b",
)

llm = ChatGroq(
    model=CHAT_MODEL,
    temperature=0,
)

print("Chat model:", CHAT_MODEL)

Chat model: openai/gpt-oss-120b


# manual query rewriting code

In [86]:
query_rewriting_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You rewrite conversational questions into clear,
standalone search queries.

Rules:
1. Do not answer the question.
2. Preserve important entities, dates, and technical terms.
3. Resolve pronouns using the conversation history.
4. Return only one rewritten query.
""",
        ),
        (
            "human",
            """
Conversation history:
{chat_history}

Current query:
{query}
""",
        ),
    ]
)

In [87]:
query_rewriting_chain = (query_rewriting_prompt| llm | StrOutputParser())

In [88]:
chat_history = """
User: How was Llama 2-Chat initially fine-tuned?
Assistant: It first underwent supervised fine-tuning.
"""

In [89]:
original_query = "What did Meta do after that?"

In [90]:
rewritten_query = query_rewriting_chain.invoke(
    {
        "chat_history": chat_history,
        "query": original_query,
    }
).strip()

In [91]:
print("Original query:")
print(original_query)

print("\nRewritten query:")
print(rewritten_query)

Original query:
What did Meta do after that?

Rewritten query:
Meta subsequent training steps for Llama 2‑Chat after supervised fine‑tuning


In [92]:
rewritten_query_documents = hybrid_retriever.invoke(
    rewritten_query
)

In [93]:
display_documents(
    rewritten_query_documents,
    title="Documents Retrieved Using the Rewritten Query",
    max_documents=5,
)


Documents Retrieved Using the Rewritten Query

RANK: 1
Paper page: 16
Section: fine_tuning
Chunk ID: llama2-page-16-chunk-65
----------------------------------------------------------------------------------------------------
Figure 9: Issues with multi-turn memory(left) can be improved with GAtt(right).
We train for between200and 400iterations for all our models, and use evaluations on held-out prompts for
earlystopping. EachiterationofPPOonthe70Bmodeltakesonaverage ≈ 330seconds. Totrainquicklywith
large batch sizes, we use FSDP (Zhao et al., 2023). This was effective when using O(1) forward or backward
passes, but caused a large slow down (≈ 20×) during generation, even when using a large batch size and KV
cache. We were able to mitigate this by consolidating the model weights to each node once before generation


RANK: 2
Paper page: 1
Section: front_matter
Chunk ID: llama2-page-1-chunk-1
-----------------------------------------------------------------------------------------------

In [94]:
class ExpandedQueryOutput(BaseModel):
    queries: List[str] = Field(
        description=(
            "Four alternative search queries expressing "
            "the same information need using different wording."
        )
    )

In [95]:
query_expansion_llm = llm.with_structured_output(ExpandedQueryOutput)

In [96]:
query_expansion_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Generate four alternative search queries for the user's query.

Use:
- synonyms,
- related technical terms,
- abbreviations where appropriate,
- alternative wording.

Do not answer the query.
Each query must preserve the original intent.
""",
        ),
        (
            "human",
            "Original query: {query}",
        ),
    ]
)

In [97]:
query_expansion_chain = (query_expansion_prompt| query_expansion_llm)

In [98]:
original_query = (
    "How was Llama 2-Chat improved using human feedback?"
)


In [99]:
expanded_output = query_expansion_chain.invoke(
    {
        "query": original_query
    }
)

In [100]:
expanded_output

ExpandedQueryOutput(queries=['What role did human feedback play in enhancing Llama 2-Chat?', 'How was Llama 2-Chat refined through user feedback?', 'In what ways did human-in-the-loop data improve Llama 2-Chat?', 'What improvements to Llama 2-Chat resulted from human annotation?'])

In [101]:
all_expanded_queries = [
    original_query,
    *expanded_output.queries,
]


In [102]:
all_expanded_queries

['How was Llama 2-Chat improved using human feedback?',
 'What role did human feedback play in enhancing Llama 2-Chat?',
 'How was Llama 2-Chat refined through user feedback?',
 'In what ways did human-in-the-loop data improve Llama 2-Chat?',
 'What improvements to Llama 2-Chat resulted from human annotation?']

In [103]:
print("Generated search queries:\n")

for number, query in enumerate(
    all_expanded_queries,
    start=1,
):
    print(f"{number}. {query}")

Generated search queries:

1. How was Llama 2-Chat improved using human feedback?
2. What role did human feedback play in enhancing Llama 2-Chat?
3. How was Llama 2-Chat refined through user feedback?
4. In what ways did human-in-the-loop data improve Llama 2-Chat?
5. What improvements to Llama 2-Chat resulted from human annotation?


In [104]:
expanded_query_documents = []

for query in all_expanded_queries:
    current_documents = dense_retriever.invoke(query)
    expanded_query_documents.extend(current_documents)

expanded_query_documents = deduplicate_documents(
    expanded_query_documents
)

display_documents(
    expanded_query_documents,
    title="Query Expansion: Combined Unique Documents",
    max_documents=10,
)


Query Expansion: Combined Unique Documents

RANK: 1
Paper page: 18
Section: fine_tuning
Chunk ID: llama2-page-18-chunk-77
----------------------------------------------------------------------------------------------------
biased in favor ofLlama 2-Chat. Therefore, for a fair comparison, we additionally compute the final results
usingGPT-4toassesswhichgenerationispreferred. TheorderinwhichChatGPTand Llama 2-Chatoutputs
appeared in GPT-4 prompt are randomly swapped to avoid any bias. As expected, the win-rate in favor of
Llama 2-Chat is less pronounced, although obtaining more than a 60% win-rate for our latestLlama 2-Chat.
The prompts correspond to a validation set of1, 586and 584prompts for safety and helpfulness, respectively.
3.4.2 Human Evaluation
Human evaluation is often considered the gold standard for ju

RANK: 2
Paper page: 5
Section: pretraining
Chunk ID: llama2-page-5-chunk-14
--------------------------------------------------------------------------------------------------

# in built langchain function to perform the multi query rewriting

In [109]:
from langchain_classic.retrievers.multi_query import (
    MultiQueryRetriever,
)

In [110]:
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=dense_retriever,
    llm=llm,
    include_original=True,
)

In [111]:
multi_query_documents = multi_query_retriever.invoke(
    "How did human feedback improve Llama 2-Chat?"
)


In [112]:
display_documents(
    multi_query_documents,
    title="Built-in MultiQueryRetriever Results",
    max_documents=10,
)


Built-in MultiQueryRetriever Results

RANK: 1
Paper page: 5
Section: pretraining
Chunk ID: llama2-page-5-chunk-14
----------------------------------------------------------------------------------------------------
Figure 4: Training ofLlama 2-Chat: This process begins with thepretraining of Llama 2 using publicly
available online sources. Following this, we create an initial version ofLlama 2-Chatthrough the application
of supervised fine-tuning. Subsequently, the model is iteratively refined using Reinforcement Learning
with Human Feedback(RLHF) methodologies, specifically through rejection sampling and Proximal Policy
Optimization (PPO). Throughout the RLHF stage, the accumulation ofiterative reward modeling datain
parallel with model enhancements is crucial to ensure the reward models remain within d

RANK: 2
Paper page: 18
Section: fine_tuning
Chunk ID: llama2-page-18-chunk-77
----------------------------------------------------------------------------------------------------
bia

In [113]:
class DecomposedQueryOutput(BaseModel):
    sub_queries: List[str] = Field(
        description=(
            "Independent and atomic search queries required "
            "to answer the complete user question."
        )
    )

In [114]:
query_decomposition_llm = llm.with_structured_output(
    DecomposedQueryOutput
)

In [115]:
query_decomposition_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Break the user's complex question into independent,
atomic search queries.

Rules:
1. Do not answer the question.
2. Generate only sub-queries needed to answer it.
3. Each sub-query must be understandable independently.
4. Preserve named entities, model names, and dates.
5. Generate between two and five sub-queries.
""",
        ),
        (
            "human",
            "Complex query: {query}",
        ),
    ]
)

In [116]:
query_decomposition_chain = (
    query_decomposition_prompt
    | query_decomposition_llm
)

In [117]:
complex_query = """
Compare Llama 2 pretraining with Llama 2-Chat fine-tuning,
and explain how Meta improved model safety.
"""

In [118]:
decomposed_output = query_decomposition_chain.invoke(
    {
        "query": complex_query
    }
)


In [119]:
sub_queries = decomposed_output.sub_queries

In [72]:
print("Original complex query:")
print(complex_query)

print("\nGenerated sub-queries:")

for number, query in enumerate(sub_queries, start=1):
    print(f"{number}. {query}")

Original complex query:

Compare Llama 2 pretraining with Llama 2-Chat fine-tuning,
and explain how Meta improved model safety.


Generated sub-queries:
1. What is the pretraining process of Llama 2?
2. What is the fine-tuning process of Llama 2-Chat?
3. How does Meta improve model safety in Llama 2 and Llama 2-Chat?


In [120]:
decomposition_results = {}

for sub_query in sub_queries:
    decomposition_results[sub_query] = hybrid_retriever.invoke(
        sub_query
    )

In [121]:
for sub_query, documents in decomposition_results.items():
    display_documents(
        documents,
        title=f"Sub-query: {sub_query}",
        max_documents=4,
    )


Sub-query: What are the data sources, architecture, and training objectives used in the pretraining of Meta's Llama 2 model?

RANK: 1
Paper page: 77
Section: appendix
Chunk ID: llama2-page-77-chunk-340
----------------------------------------------------------------------------------------------------
Llama 2.
Hardware and Software(Section 2.2)
Training Factors We used custom training libraries, Meta’s Research Super Cluster, and produc-
tion clusters for pretraining. Fine-tuning, annotation, and evaluation were also
performed on third-party cloud compute.
Carbon Footprint Pretraining utilized a cumulative 3.3M GPU hours of computation on hardware
of type A100-80GB (TDP of 350-400W). Estimated total emissions were 539
tCO2eq, 100% of which were offset by Meta’s sustainability program.
Training Data(Sections 2.1 and 3)
Overview Llama 2was pretrained on 2 trillion tokens of data from publicly

RANK: 2
Paper page: 7
Section: pretraining
Chunk ID: llama2-page-7-chunk-23
------------------

In [122]:
all_decomposition_documents = []

for documents in decomposition_results.values():
    all_decomposition_documents.extend(documents)

all_decomposition_documents = deduplicate_documents(
    all_decomposition_documents
)

display_documents(
    all_decomposition_documents,
    title="Combined Evidence from All Sub-Queries",
    max_documents=12,
)


Combined Evidence from All Sub-Queries

RANK: 1
Paper page: 77
Section: appendix
Chunk ID: llama2-page-77-chunk-340
----------------------------------------------------------------------------------------------------
Llama 2.
Hardware and Software(Section 2.2)
Training Factors We used custom training libraries, Meta’s Research Super Cluster, and produc-
tion clusters for pretraining. Fine-tuning, annotation, and evaluation were also
performed on third-party cloud compute.
Carbon Footprint Pretraining utilized a cumulative 3.3M GPU hours of computation on hardware
of type A100-80GB (TDP of 350-400W). Estimated total emissions were 539
tCO2eq, 100% of which were offset by Meta’s sustainability program.
Training Data(Sections 2.1 and 3)
Overview Llama 2was pretrained on 2 trillion tokens of data from publicly

RANK: 2
Paper page: 7
Section: pretraining
Chunk ID: llama2-page-7-chunk-23
----------------------------------------------------------------------------------------------------
We 

Complex query
      ↓
Atomic sub-queries
      ↓
Retrieve for every sub-query
      ↓
Merge evidence
      ↓
Remove duplicates

## HyDE (Hypothetical Document Embeddings)

HyDE asks an LLM to write a plausible passage that could answer the query. The hypothetical passage is embedded and used for dense retrieval instead of embedding the short user query directly. The generated text is only a search aid; the final answer must still be grounded in retrieved source documents.

In [123]:
from typing import Any

from langchain_core.retrievers import BaseRetriever


class HyDERetriever(BaseRetriever):
    """Generate a hypothetical passage and retrieve real documents with it."""

    hypothesis_chain: Any
    vector_retriever: Any
    last_hypothetical_document: str = ""

    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager,
    ):
        hypothetical_document = self.hypothesis_chain.invoke(
            {"query": query}
        ).strip()

        self.last_hypothetical_document = hypothetical_document

        return self.vector_retriever.invoke(
            hypothetical_document,
            config={
                "callbacks": run_manager.get_child(),
            },
        )

In [124]:
hyde_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Write a concise research-paper passage that would directly answer the user's question.
Use technical terminology and named entities likely to appear in the source document.
Do not mention that the passage is hypothetical.
Do not add citations, headings, or commentary.
Return only the passage.
""",
        ),
        ("human", "Question: {query}"),
    ]
)

hyde_hypothesis_chain = (
    hyde_prompt
    | llm
    | StrOutputParser()
)

hyde_dense_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

hyde_retriever = HyDERetriever(
    hypothesis_chain=hyde_hypothesis_chain,
    vector_retriever=hyde_dense_retriever,
)

In [125]:
hyde_query = (
    "How did Meta use human feedback to improve the helpfulness "
    "and safety of Llama 2-Chat?"
)

hyde_documents = hyde_retriever.invoke(hyde_query)

print("Original query:")
print(hyde_query)

print("\nHypothetical document used for retrieval:")
print(hyde_retriever.last_hypothetical_document)

display_documents(
    hyde_documents,
    title="HyDE Retrieval Results",
    max_documents=5,
)

Original query:
How did Meta use human feedback to improve the helpfulness and safety of Llama 2-Chat?

Hypothetical document used for retrieval:
Meta improved Llama 2‑Chat’s helpfulness and safety by integrating a multi‑stage human‑feedback loop into its fine‑tuning pipeline. First, a curated set of high‑quality instruction–response pairs was generated through supervised fine‑tuning (SFT) using expert annotators who rated responses for relevance, factuality, and tone. These annotations were then used to train a reward model that predicts human preference scores. The reward model guided a reinforcement learning from human feedback (RLHF) stage, where policy gradients adjusted the language model to maximize predicted preference while respecting safety constraints. Parallel to RLHF, a dedicated safety‑fine‑tuning phase employed red‑team annotations that labeled toxic, biased, or disallowed content; a binary safety classifier was trained on this data and incorporated as a penalty term dur

User query
     ↓
LLM-generated hypothetical document
     ↓
Embed the hypothetical document
     ↓
Dense vector search
     ↓
Real source documents

## HyDE with LangChain's built-in helper

`HypotheticalDocumentEmbedder` is an embeddings wrapper, not a complete retriever. For a query, it generates a hypothetical document with the LLM and embeds that generated text. Chroma then uses the resulting vector to search the collection of real PDF chunks.

In [1]:
try:
    # LangChain v1: legacy chains live in langchain-classic.
    from langchain_classic.chains.hyde.base import (
        HypotheticalDocumentEmbedder,
    )
except ImportError:
    # Compatibility fallback for older LangChain versions.
    from langchain.chains.hyde.base import (
        HypotheticalDocumentEmbedder,
    )

d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
built_in_hyde_embeddings = (
    HypotheticalDocumentEmbedder.from_llm(
        llm=llm,
        base_embeddings=embeddings,
        prompt_key="web_search",
    )
)

# Reuse the existing collection. Real PDF chunks remain embedded with
# `embeddings`; HyDE is used only to transform query embeddings.
built_in_hyde_vector_store = Chroma(
    collection_name="llama2_retriever_demo",
    embedding_function=built_in_hyde_embeddings,
    persist_directory=str(PERSIST_DIRECTORY),
)

built_in_hyde_retriever = (
    built_in_hyde_vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 5},
    )
)

NameError: name 'llm' is not defined

In [ ]:
built_in_hyde_query = (
    "How did Meta use human feedback to improve the helpfulness "
    "and safety of Llama 2-Chat?"
)

built_in_hyde_documents = (
    built_in_hyde_retriever.invoke(
        built_in_hyde_query
    )
)

display_documents(
    built_in_hyde_documents,
    title="Built-in LangChain HyDE Results",
    max_documents=5,
)

> **Note:** Do not use `built_in_hyde_embeddings` when initially indexing the PDF chunks. Index real documents with the normal `OpenAIEmbeddings` object, then use the HyDE wrapper only when loading the collection for retrieval.

In [76]:
candidate_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 20
    },
)

Vector search
     ↓
Top 20 candidates

In [77]:
cross_encoder = HuggingFaceCrossEncoder(
    model_name="cross-encoder/ms-marco-MiniLM-L6-v2",
    model_kwargs={
        "device": "cpu"
    },
)

d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sunny\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 13097.84it/s]


In [78]:
cross_encoder_reranker = CrossEncoderReranker(
    model=cross_encoder,
    top_n=5,
)

In [79]:
reranking_retriever = ContextualCompressionRetriever(
    base_retriever=candidate_retriever,
    base_compressor=cross_encoder_reranker,
)

In [80]:
reranking_query = (
    "How did Meta collect and use human preference data to train Llama 2-Chat?"
)

In [81]:
initial_candidates = candidate_retriever.invoke(
    reranking_query
)

In [82]:
initial_candidates

[Document(id='577892fb-3838-4c4d-9978-4488765bf837', metadata={'producer': 'pdfTeX-1.40.25', 'year': 2023, 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'creationdate': '2023-07-20T00:30:36+00:00', 'paper_page': 10, 'paper': 'Llama 2', 'title': '', 'page': 9, 'keywords': '', 'section': 'fine_tuning', 'chunk_id': 'llama2-page-10-chunk-38', 'author': '', 'document_type': 'research_paper', 'moddate': '2023-07-20T00:30:36+00:00', 'total_pages': 77, 'subject': '', 'creator': 'LaTeX with hyperref', 'start_index': 2384, 'page_label': '10', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-36-29-July-2026-Retriever\\data\\llama2-research-paper.pdf', 'trapped': '/False', 'organization': 'Meta', 'access_level': 'public'}, page_content='can be found in Section 4.2.1.\nHuman annotations were collected in batches on a weekly basis. As we collected more preference data, our\nreward models improved, and we were abl

In [83]:
display_documents(
    initial_candidates,
    title="Before Reranking: Initial Vector Candidates",
    max_documents=10,
)


Before Reranking: Initial Vector Candidates

RANK: 1
Paper page: 10
Section: fine_tuning
Chunk ID: llama2-page-10-chunk-38
----------------------------------------------------------------------------------------------------
can be found in Section 4.2.1.
Human annotations were collected in batches on a weekly basis. As we collected more preference data, our
reward models improved, and we were able to train progressively better versions forLlama 2-Chat (see
the results in Section 5, Figure 20).Llama 2-Chat improvement also shifted the model’s data distribution.
Since reward model accuracy can quickly degrade if not exposed to this new sample distribution, i.e., from
hyper-specialization (Scialom et al., 2020b), it is important before a newLlama 2-Chat tuning iteration to
gather new preference data using the latest

RANK: 2
Paper page: 5
Section: pretraining
Chunk ID: llama2-page-5-chunk-14
-------------------------------------------------------------------------------------------------

In [84]:
reranked_documents = reranking_retriever.invoke(reranking_query)

In [85]:
display_documents(
    reranked_documents,
    title="After Reranking: Final Top Documents",
    max_documents=5,
)


After Reranking: Final Top Documents

RANK: 1
Paper page: 10
Section: fine_tuning
Chunk ID: llama2-page-10-chunk-38
----------------------------------------------------------------------------------------------------
can be found in Section 4.2.1.
Human annotations were collected in batches on a weekly basis. As we collected more preference data, our
reward models improved, and we were able to train progressively better versions forLlama 2-Chat (see
the results in Section 5, Figure 20).Llama 2-Chat improvement also shifted the model’s data distribution.
Since reward model accuracy can quickly degrade if not exposed to this new sample distribution, i.e., from
hyper-specialization (Scialom et al., 2020b), it is important before a newLlama 2-Chat tuning iteration to
gather new preference data using the latest

RANK: 2
Paper page: 11
Section: fine_tuning
Chunk ID: llama2-page-11-chunk-45
----------------------------------------------------------------------------------------------------
t

User query
     ↓
Dense Retriever
     ↓
Top 20 candidates
     ↓
Cross-Encoder Reranker
     ↓
Query-document relevance evaluation
     ↓
Final top 5 documents

In [86]:
bm25_retriever.k = 15

In [87]:
dense_candidate_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 15
    },
)

In [88]:
hybrid_candidate_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        dense_candidate_retriever,
    ],
    weights=[
        0.5,
        0.5,
    ],
)

In [89]:
hybrid_reranking_retriever = ContextualCompressionRetriever(
    base_retriever=hybrid_candidate_retriever,
    base_compressor=cross_encoder_reranker,
)

In [90]:
query = (
    "What techniques did Meta use to improve the helpfulness and safety of Llama 2-Chat?"
)

In [91]:
final_documents = hybrid_reranking_retriever.invoke(query)

In [92]:
final_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-36-29-July-2026-Retriever\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public', 'start_index': 823, 'chunk_id': 'llama2-page-1-chunk-1'}, page_content='Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang\nAngela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojnic\nSergey Edunov Thomas Scialom∗\nGenAI, Meta\nAbstract\nIn t

In [93]:
display_documents(
    final_documents,
    title="Hybrid Retrieval + Cross-Encoder Reranking",
    max_documents=5,
)


Hybrid Retrieval + Cross-Encoder Reranking

RANK: 1
Paper page: 1
Section: front_matter
Chunk ID: llama2-page-1-chunk-1
----------------------------------------------------------------------------------------------------
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojnic
Sergey Edunov Thomas Scialom∗
GenAI, Meta
Abstract
In this work, we develop and release Llama 2, a collection of pretrained and fine-tuned
large language models (LLMs) ranging in scale from 7 billion to 70 billion parameters.
Our fine-tuned LLMs, calledLlama 2-Chat, are optimized for dialogue use cases. Our
models outperform open-source chat models on most benchmarks we tested, and based on
our human evaluations for helpfulness and 

RANK: 2
Paper page: 3
Section: introduction
Chunk ID: llama2-page-3-chunk-9
----------------------------------------------------------------------------------------------------

                         ┌── BM25 Search ───────┐
User query ──────────────┤                      ├── Weighted RRF
                         └── Dense Search ──────┘
                                                   ↓
                                          Candidate documents
                                                   ↓
                                       Cross-Encoder Reranker
                                                   ↓
                                           Final top documents

Sparse Retrieval
    = BM25Retriever

Dense Retrieval
    = VectorStoreRetriever

Hybrid Retrieval
    = BM25 + Dense + EnsembleRetriever + Weighted RRF

Query Rewriting
    = Convert a contextual query into a standalone query

Query Expansion
    = Generate related query variations and merge their results

Query Decomposition
    = Split one complex query into atomic searchable queries

HyDE
    = Generate a hypothetical answer passage and use its embedding for dense retrieval

Reranking
    = Retrieve broad candidates and reorder them using a cross-encoder

1. BM25 sparse retrieval
2. Dense vector retrieval
3. Hybrid retrieval
4. Query rewriting
5. Query expansion
6. Query decomposition
7. HyDE retrieval
8. Dense retrieval + reranking
9. Hybrid retrieval + reranking

HOME_WORK
weighted fusion
resiporcal rank fusion
multi query retriever
Multi hop retriever
parenet document retriever
sentence window retriever
contextual compression

CUSTOM_RETRIEVER
Langchain 
langchain provide one baseretriever class ontop of it you can create custom retriever with your own logic(when you are usingh langchain/langgraph)

In [ ]:
i will take one more hour reteriever
will discuss about the prompting

In [ ]:
# from langchain_openai import ChatOpenAI
# from langchain_core.prompts import ChatPromptTemplate

# llm = ChatOpenAI(
#     model="gpt-4.1-mini",
#     temperature=0
# )

# hyde_prompt = ChatPromptTemplate.from_messages(
#     [
#         (
#             "system",
#             """
# Generate a short hypothetical document that could answer
# the user's question.

# Do not mention that the document is hypothetical.
# Write it in the style of a factual knowledge-base passage.
# """
#         ),
#         (
#             "human",
#             "{query}"
#         )
#     ]
# )

# query = "How does Llama 2 improve safety?"

# response = llm.invoke(
#     hyde_prompt.format_messages(
#         query=query
#     )
# )

# hypothetical_document = response.content

# print("Original Query:")
# print(query)

# print("\nHypothetical Document:")
# print(hypothetical_document)


# hyde_vector = base_embeddings.embed_query(
#     hypothetical_document
# )

# hyde_documents = vector_store.similarity_search_by_vector(
#     hyde_vector,
#     k=4
# )

# for i, document in enumerate(hyde_documents, start=1):
#     print("=" * 80)
#     print(f"RESULT {i}")
#     print(document.page_content[:1000])

multimodal RAG builder
full flede project

MEGA ASSISGNMENT on Sunday